# Multi-Route Model v1.1 with Confidence Intervals

This notebook extends the best model configuration (from 8b) with **prediction confidence intervals**, which quantifies the uncertainty level of the predictions. In this notebook, only the confidence interval of the Ridge model is computed.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import date
from calendar import monthrange
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, f1_score, roc_auc_score, precision_score, recall_score,
    confusion_matrix
)
import warnings
warnings.filterwarnings('ignore')

import holidays

plt.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.linewidth'] = 1.5

try:
    import xgboost as xgb
    HAS_XGB = True
    print("XGBoost available")
except ImportError:
    HAS_XGB = False
    print("XGBoost not installed")

%matplotlib inline

## 1. Data Preparation

In [ ]:
# Load multi-route data
df = pd.read_csv('../data/processed/ml_training_data_multiroute_hols.csv')

df['year_month_dt'] = pd.to_datetime(df['year_month'])
df['month_num'] = df['year_month_dt'].dt.month
df['year'] = df['year'].astype(int)
df['airline_route'] = df['airline'] + '_' + df['departing_port'] + '_' + df['arriving_port']
df['route'] = df['departing_port'] + '_' + df['arriving_port']
df = df.sort_values(['airline_route', 'year_month_dt']).reset_index(drop=True)

print(f"Original shape: {df.shape}")
print(f"Date range: {df['year_month'].min()} to {df['year_month'].max()}")

## 2. Filter Low-Volume Airline-Routes

In [ ]:
volume_threshold = 50
airline_route_volume = df.groupby('airline_route')['sectors_scheduled'].mean().reset_index()
airline_route_volume.columns = ['airline_route', 'avg_volume']
high_volume_ar = airline_route_volume[airline_route_volume['avg_volume'] >= volume_threshold]['airline_route'].tolist()

# Show excluded routes
excluded = airline_route_volume[airline_route_volume['avg_volume'] < volume_threshold].sort_values('avg_volume')
print(f"Volume threshold: {volume_threshold} flights/month")
print(f"\nExcluded airline-routes ({len(excluded)}):")
for _, row in excluded.iterrows():
    print(f"  {row['airline_route']}: {row['avg_volume']:.0f} flights/mo")

df_filtered = df[df['airline_route'].isin(high_volume_ar)].copy()
print(f"\nRecords before filtering: {len(df)}")
print(f"Records after filtering:  {len(df_filtered)}")

In [ ]:
# Exclude Melbourne-Hobart route & Adelaide-Sydney (found after running this notebook)
# Reason: Anomalous 2019 data causes unreliable predictions (see notebook 6c)
anomalous_routes = ['Melbourne_Hobart', 'Adelaide-Sydney']
records_before = len(df_filtered)
df_filtered = df_filtered[~df_filtered['route'].isin(anomalous_routes)].copy()
print(f"\nExcluded anomalous routes: {anomalous_routes}")
print(f"Records before: {records_before}")
print(f"Records after:  {len(df_filtered)}")

## 3. Feature Engineering

In [ ]:
# Lagged features
df_filtered['delay_rate_lag1'] = df_filtered.groupby('airline_route')['delay_rate'].shift(1)
df_filtered['delay_rate_lag2'] = df_filtered.groupby('airline_route')['delay_rate'].shift(2)
df_filtered['delay_rate_gradient'] = df_filtered['delay_rate_lag1'] - df_filtered['delay_rate_lag2']

# Cyclical month encoding
df_filtered['month_sin'] = np.sin(2 * np.pi * df_filtered['month_num'] / 12)
df_filtered['month_cos'] = np.cos(2 * np.pi * df_filtered['month_num'] / 12)

# One-hot encoding
airline_dummies = pd.get_dummies(df_filtered['airline'], prefix='airline')
df_filtered = pd.concat([df_filtered, airline_dummies], axis=1)
airline_cols = list(airline_dummies.columns)

route_dummies = pd.get_dummies(df_filtered['route'], prefix='route')
df_filtered = pd.concat([df_filtered, route_dummies], axis=1)
route_cols = list(route_dummies.columns)

# Weather features with transformations
df_filtered['rainy_days_arr_exp'] = np.exp(df_filtered['rainy_days_arr'] / df_filtered['rainy_days_arr'].max())
df_filtered['temp_volatility_total'] = df_filtered['temp_volatility_dep'] + df_filtered['temp_volatility_arr']
df_filtered['temp_volatility_total_exp'] = np.exp(df_filtered['temp_volatility_total'] / df_filtered['temp_volatility_total'].max())
df_filtered['extreme_weather_days_total'] = df_filtered['extreme_weather_days_dep'] + df_filtered['extreme_weather_days_arr']

# Drop NaN (from lag features)
df_clean = df_filtered.dropna(subset=['delay_rate_lag1', 'delay_rate_lag2', 'delay_rate_gradient']).copy()
print(f"Rows after dropping NaN: {len(df_clean)}")

## 4. Train/Validation/Test Split

In [ ]:
#train_mask = ((df_clean['year'] <= 2017) | (df_clean['year'] == 2024))
train_mask = (((df_clean['year'] >=2010) & (df_clean['year'] <= 2017)) | (df_clean['year'] == 2023))
val_mask = ((df_clean['year'] == 2018) | (df_clean['year'] == 2024))
test_mask = ((df_clean['year'] == 2019) | (df_clean['year'] >= 2025))

print(f"Train (2010-2017, 2023): {train_mask.sum()} samples")
print(f"Validation (2018, 2024): {val_mask.sum()} samples")
print(f"Test (2019, 2025):       {test_mask.sum()} samples")

In [ ]:
# Define final feature set
base_features = airline_cols + route_cols + ['month_sin', 'month_cos', 'delay_rate_lag1', 'sectors_scheduled']
weather_features = ['rainy_days_arr_exp', 'delay_rate_gradient', 'temp_volatility_total_exp', 'extreme_weather_days_total']
holiday_features = ['n_public_holidays_total', 'pct_school_holiday']

FINAL_FEATURES = base_features + weather_features + holiday_features

print(f"Total features: {len(FINAL_FEATURES)}")
print(f"\nFeature breakdown:")
print(f"  - Airline encoding: {len(airline_cols)}")
print(f"  - Route encoding:   {len(route_cols)}")
print(f"  - Month encoding:   2 (sin, cos)")
print(f"  - Lag features:     2 (lag1, gradient)")
print(f"  - Volume:           1 (sectors_scheduled)")
print(f"  - Weather:          3 (rainy, temp_volatility, extreme)")
print(f"  - Holidays:         2 (public, school)")

In [ ]:
# Prepare data
X_train = df_clean.loc[train_mask, FINAL_FEATURES].values
X_val = df_clean.loc[val_mask, FINAL_FEATURES].values
X_test = df_clean.loc[test_mask, FINAL_FEATURES].values

y_train_reg = df_clean.loc[train_mask, 'delay_rate'].values
y_val_reg = df_clean.loc[val_mask, 'delay_rate'].values
y_test_reg = df_clean.loc[test_mask, 'delay_rate'].values

y_train_clf = df_clean.loc[train_mask, 'is_high_delay'].values
y_val_clf = df_clean.loc[val_mask, 'is_high_delay'].values
y_test_clf = df_clean.loc[test_mask, 'is_high_delay'].values

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Data prepared.")
print(f"X_train: {X_train.shape}")
print(f"X_val:   {X_val.shape}")
print(f"X_test:  {X_test.shape}")

## 5. Regression Models

In [ ]:
reg_results = []

# Ridge Regression
print("Training Ridge Regression...")
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train_reg)

ridge_val_pred = ridge.predict(X_val_scaled)
ridge_test_pred = ridge.predict(X_test_scaled)

ridge_val_r2 = r2_score(y_val_reg, ridge_val_pred)
ridge_test_r2 = r2_score(y_test_reg, ridge_test_pred)
ridge_test_rmse = np.sqrt(mean_squared_error(y_test_reg, ridge_test_pred))
ridge_test_mae = mean_absolute_error(y_test_reg, ridge_test_pred)

print(f"  Val R²:   {ridge_val_r2:.4f}")
print(f"  Test R²:  {ridge_test_r2:.4f}")
print(f"  Test RMSE: {ridge_test_rmse:.4f}")
print(f"  Test MAE:  {ridge_test_mae:.4f}")

reg_results.append({'Model': 'Ridge', 'Val_R2': ridge_val_r2, 'Test_R2': ridge_test_r2, 
                    'Test_RMSE': ridge_test_rmse, 'Test_MAE': ridge_test_mae})

In [ ]:
# Random Forest Regression
print("Training Random Forest Regression...")
rf_reg = RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
rf_reg.fit(X_train, y_train_reg)

rf_val_pred = rf_reg.predict(X_val)
rf_test_pred = rf_reg.predict(X_test)

rf_val_r2 = r2_score(y_val_reg, rf_val_pred)
rf_test_r2 = r2_score(y_test_reg, rf_test_pred)
rf_test_rmse = np.sqrt(mean_squared_error(y_test_reg, rf_test_pred))
rf_test_mae = mean_absolute_error(y_test_reg, rf_test_pred)

print(f"  Val R²:   {rf_val_r2:.4f}")
print(f"  Test R²:  {rf_test_r2:.4f}")
print(f"  Test RMSE: {rf_test_rmse:.4f}")
print(f"  Test MAE:  {rf_test_mae:.4f}")

reg_results.append({'Model': 'Random Forest', 'Val_R2': rf_val_r2, 'Test_R2': rf_test_r2,
                    'Test_RMSE': rf_test_rmse, 'Test_MAE': rf_test_mae})

## 6. Classification Models

In [ ]:
clf_results = []

# Logistic Regression
print("Training Logistic Regression...")
logreg = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
logreg.fit(X_train_scaled, y_train_clf)

logreg_val_pred = logreg.predict(X_val_scaled)
logreg_val_proba = logreg.predict_proba(X_val_scaled)[:, 1]
logreg_test_pred = logreg.predict(X_test_scaled)
logreg_test_proba = logreg.predict_proba(X_test_scaled)[:, 1]

logreg_val_f1 = f1_score(y_val_clf, logreg_val_pred)
logreg_test_f1 = f1_score(y_test_clf, logreg_test_pred)
logreg_test_auc = roc_auc_score(y_test_clf, logreg_test_proba)
logreg_test_precision = precision_score(y_test_clf, logreg_test_pred)
logreg_test_recall = recall_score(y_test_clf, logreg_test_pred)

print(f"  Val F1:       {logreg_val_f1:.4f}")
print(f"  Test F1:      {logreg_test_f1:.4f}")
print(f"  Test AUC:     {logreg_test_auc:.4f}")
print(f"  Test Precision: {logreg_test_precision:.4f}")
print(f"  Test Recall:    {logreg_test_recall:.4f}")

clf_results.append({'Model': 'Logistic', 'Val_F1': logreg_val_f1, 'Test_F1': logreg_test_f1,
                    'Test_AUC': logreg_test_auc, 'Test_Precision': logreg_test_precision, 
                    'Test_Recall': logreg_test_recall})

In [ ]:
# Random Forest Classification
print("Training Random Forest Classification...")
rf_clf = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
rf_clf.fit(X_train, y_train_clf)

rf_clf_val_pred = rf_clf.predict(X_val)
rf_clf_val_proba = rf_clf.predict_proba(X_val)[:, 1]
rf_clf_test_pred = rf_clf.predict(X_test)
rf_clf_test_proba = rf_clf.predict_proba(X_test)[:, 1]

rf_clf_val_f1 = f1_score(y_val_clf, rf_clf_val_pred)
rf_clf_test_f1 = f1_score(y_test_clf, rf_clf_test_pred)
rf_clf_test_auc = roc_auc_score(y_test_clf, rf_clf_test_proba)
rf_clf_test_precision = precision_score(y_test_clf, rf_clf_test_pred)
rf_clf_test_recall = recall_score(y_test_clf, rf_clf_test_pred)

print(f"  Val F1:       {rf_clf_val_f1:.4f}")
print(f"  Test F1:      {rf_clf_test_f1:.4f}")
print(f"  Test AUC:     {rf_clf_test_auc:.4f}")
print(f"  Test Precision: {rf_clf_test_precision:.4f}")
print(f"  Test Recall:    {rf_clf_test_recall:.4f}")

clf_results.append({'Model': 'Random Forest', 'Val_F1': rf_clf_val_f1, 'Test_F1': rf_clf_test_f1,
                    'Test_AUC': rf_clf_test_auc, 'Test_Precision': rf_clf_test_precision,
                    'Test_Recall': rf_clf_test_recall})

In [ ]:
# XGBoost Classification
if HAS_XGB:
    print("Training XGBoost Classification...")
    xgb_clf = xgb.XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, 
                                 min_child_weight=5, random_state=42, n_jobs=-1)
    xgb_clf.fit(X_train, y_train_clf, eval_set=[(X_val, y_val_clf)], verbose=False)
    
    xgb_val_pred = xgb_clf.predict(X_val)
    xgb_val_proba = xgb_clf.predict_proba(X_val)[:, 1]
    xgb_test_pred = xgb_clf.predict(X_test)
    xgb_test_proba = xgb_clf.predict_proba(X_test)[:, 1]
    
    xgb_val_f1 = f1_score(y_val_clf, xgb_val_pred)
    xgb_test_f1 = f1_score(y_test_clf, xgb_test_pred)
    xgb_test_auc = roc_auc_score(y_test_clf, xgb_test_proba)
    xgb_test_precision = precision_score(y_test_clf, xgb_test_pred)
    xgb_test_recall = recall_score(y_test_clf, xgb_test_pred)
    
    print(f"  Val F1:       {xgb_val_f1:.4f}")
    print(f"  Test F1:      {xgb_test_f1:.4f}")
    print(f"  Test AUC:     {xgb_test_auc:.4f}")
    print(f"  Test Precision: {xgb_test_precision:.4f}")
    print(f"  Test Recall:    {xgb_test_recall:.4f}")
    
    clf_results.append({'Model': 'XGBoost', 'Val_F1': xgb_val_f1, 'Test_F1': xgb_test_f1,
                        'Test_AUC': xgb_test_auc, 'Test_Precision': xgb_test_precision,
                        'Test_Recall': xgb_test_recall})

## 7. Results Summary

In [ ]:
reg_df = pd.DataFrame(reg_results)
clf_df = pd.DataFrame(clf_results)

print("="*80)
print("REGRESSION RESULTS")
print("="*80)
print(f"\n{'Model':<15} {'Val R²':>10} {'Test R²':>10} {'Test RMSE':>12} {'Test MAE':>10}")
print("-" * 60)
for _, row in reg_df.iterrows():
    print(f"{row['Model']:<15} {row['Val_R2']:>10.4f} {row['Test_R2']:>10.4f} {row['Test_RMSE']:>12.4f} {row['Test_MAE']:>10.4f}")

print("\n" + "="*80)
print("CLASSIFICATION RESULTS")
print("="*80)
print(f"\n{'Model':<15} {'Val F1':>10} {'Test F1':>10} {'Test AUC':>10} {'Precision':>10} {'Recall':>10}")
print("-" * 70)
for _, row in clf_df.iterrows():
    print(f"{row['Model']:<15} {row['Val_F1']:>10.4f} {row['Test_F1']:>10.4f} {row['Test_AUC']:>10.4f} {row['Test_Precision']:>10.4f} {row['Test_Recall']:>10.4f}")

## 8. Actual vs Predicted

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Ridge
ax = axes[0]
ax.scatter(y_test_reg, ridge_test_pred, alpha=0.4, s=20)
ax.plot([0, 0.6], [0, 0.6], 'r--', label='Perfect prediction')
ax.set_xlabel('Actual Delay Rate')
ax.set_ylabel('Predicted Delay Rate')
ax.set_title(f'Ridge Regression (R² = {ridge_test_r2:.3f})')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 0.6)
ax.set_ylim(0, 0.6)

# Random Forest
ax = axes[1]
ax.scatter(y_test_reg, rf_test_pred, alpha=0.4, s=20)
ax.plot([0, 0.6], [0, 0.6], 'r--', label='Perfect prediction')
ax.set_xlabel('Actual Delay Rate')
ax.set_ylabel('Predicted Delay Rate')
ax.set_title(f'Random Forest (R² = {rf_test_r2:.3f})')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 0.6)
ax.set_ylim(0, 0.6)

plt.tight_layout()
plt.show()

## 9. Confidence Intervals

Compute 90% prediction intervals using route-specific residuals method:
- **Route-specific residuals**: Estimate uncertainty from validation set errors for each route

In [ ]:
# Method 1: Bootstrap-Based Confidence Intervals for Ridge
# Use residuals from validation set to estimate prediction uncertainty

# Compute residuals on validation set
val_residuals = y_val_reg - ridge_val_pred

# Estimate prediction intervals using percentile method (90% CI)
# Assume residuals are approximately normally distributed
residual_std = np.std(val_residuals)
z_90 = 1.645  # For 90% CI (one-sided 5%)

# Compute global CI bounds
ridge_ci_halfwidth_global = z_90 * residual_std
ridge_lower_test = ridge_test_pred - ridge_ci_halfwidth_global
ridge_upper_test = ridge_test_pred + ridge_ci_halfwidth_global
ridge_ci_width_test = 2 * ridge_ci_halfwidth_global * np.ones_like(ridge_test_pred)

print("Ridge 90% CI (Global Residual Method):")
print(f"  Validation set residual std: {residual_std:.4f}")
print(f"  Test set mean CI width: {ridge_ci_width_test.mean():.4f}")
print(f"  CI halfwidth: ±{ridge_ci_halfwidth_global:.4f}")

In [ ]:
# Method 2: Route-Specific Residual-Based Intervals
# Compute residuals on validation set to estimate route-specific uncertainty

df_val = df_clean[val_mask].copy()
df_val['ridge_pred'] = ridge_val_pred
df_val['residual'] = df_val['delay_rate'] - df_val['ridge_pred']

# Compute standard deviation of residuals by route
route_residual_stats = df_val.groupby('route')['residual'].agg(['mean', 'std', 'count']).reset_index()
route_residual_stats.columns = ['route', 'residual_mean', 'residual_std', 'n_val']

# For 90% CI: use 1.645 * std (normal approximation)
route_residual_stats['ci_halfwidth'] = 1.645 * route_residual_stats['residual_std']

print("Route-Specific Residual Statistics (Validation Set):")
print("-" * 70)
print(f"{'Route':<22} {'Residual Mean':>14} {'Residual Std':>14} {'90% CI ±':>12}")
print("-" * 70)
for _, row in route_residual_stats.sort_values('residual_std', ascending=False).iterrows():
    print(f"{row['route']:<22} {row['residual_mean']:>14.4f} {row['residual_std']:>14.4f} {row['ci_halfwidth']:>12.4f}")

In [ ]:
# Create test dataframe
df_test = df_clean[test_mask].copy()
print(f"Test set size: {len(df_test)} samples")
print()

# Apply route-specific CIs to test set
df_test['ridge_pred'] = ridge_test_pred
df_test['ridge_lower_global'] = ridge_lower_test
df_test['ridge_upper_global'] = ridge_upper_test
df_test['ridge_ci_width_global'] = ridge_ci_width_test

# Merge route-specific residual stats
df_test = df_test.merge(route_residual_stats[['route', 'residual_std', 'ci_halfwidth']], on='route', how='left')

# Compute route-specific CIs
df_test['ridge_lower_route'] = df_test['ridge_pred'] - df_test['ci_halfwidth']
df_test['ridge_upper_route'] = df_test['ridge_pred'] + df_test['ci_halfwidth']

# Clip CIs to valid range [0, 1]
df_test['ridge_lower_global'] = df_test['ridge_lower_global'].clip(0, 1)
df_test['ridge_upper_global'] = df_test['ridge_upper_global'].clip(0, 1)
df_test['ridge_lower_route'] = df_test['ridge_lower_route'].clip(0, 1)
df_test['ridge_upper_route'] = df_test['ridge_upper_route'].clip(0, 1)

print("Confidence intervals applied to test set.")

In [ ]:
# Coverage analysis: What percentage of actuals fall within the predicted CIs?

# Global CI coverage
coverage_global = ((df_test['delay_rate'] >= df_test['ridge_lower_global']) & 
                   (df_test['delay_rate'] <= df_test['ridge_upper_global'])).mean()

# Route-specific CI coverage
coverage_route = ((df_test['delay_rate'] >= df_test['ridge_lower_route']) & 
                  (df_test['delay_rate'] <= df_test['ridge_upper_route'])).mean()

print("90% CI Coverage Analysis (Test Set):")
print("=" * 50)
print(f"  Global method:            {coverage_global*100:.1f}%")
print(f"  Route-specific method:    {coverage_route*100:.1f}%")
print(f"  Target coverage:          90.0%")
print()
if coverage_route < 0.85:
    print("  ⚠️  Route-specific CIs may be too narrow (undercoverage)")
elif coverage_route > 0.95:
    print("  ⚠️  Route-specific CIs may be too wide (overcoverage)")
else:
    print("  ✓ Route-specific CIs are well-calibrated")

In [ ]:
# Visualise predictions with confidence intervals for a sample of data points
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sample 50 points for clarity
sample_idx = np.random.RandomState(42).choice(len(df_test), size=min(50, len(df_test)), replace=False)
sample_idx = np.sort(sample_idx)

# Global CI
ax = axes[0]
ax.errorbar(range(len(sample_idx)), df_test.iloc[sample_idx]['ridge_pred'], 
            yerr=[df_test.iloc[sample_idx]['ridge_pred'] - df_test.iloc[sample_idx]['ridge_lower_global'],
                  df_test.iloc[sample_idx]['ridge_upper_global'] - df_test.iloc[sample_idx]['ridge_pred']],
            fmt='o', capsize=3, alpha=0.6, label='Predicted ± 90% CI', markersize=4)
ax.scatter(range(len(sample_idx)), df_test.iloc[sample_idx]['delay_rate'], 
           color='red', s=20, zorder=5, label='Actual')
ax.set_xlabel('Sample Index')
ax.set_ylabel('Delay Rate')
ax.set_title(f'Global CI (Coverage: {coverage_global*100:.1f}%)')
ax.legend()
ax.grid(True, alpha=0.3)

# Route-specific CI
ax = axes[1]
ax.errorbar(range(len(sample_idx)), df_test.iloc[sample_idx]['ridge_pred'],
            yerr=[df_test.iloc[sample_idx]['ridge_pred'] - df_test.iloc[sample_idx]['ridge_lower_route'],
                  df_test.iloc[sample_idx]['ridge_upper_route'] - df_test.iloc[sample_idx]['ridge_pred']],
            fmt='o', capsize=3, alpha=0.6, label='Predicted ± 90% CI', markersize=4)
ax.scatter(range(len(sample_idx)), df_test.iloc[sample_idx]['delay_rate'],
           color='red', s=20, zorder=5, label='Actual')
ax.set_xlabel('Sample Index')
ax.set_ylabel('Delay Rate')
ax.set_title(f'Route-Specific CI (Coverage: {coverage_route*100:.1f}%)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 12. Prediction Reliability by Route

Rank routes by prediction uncertainty and identify which routes have reliable vs unreliable predictions.

In [ ]:
# Compute per-route CI statistics and coverage
route_ci_stats = []

for route in sorted(df_test['route'].unique()):
    route_data = df_test[df_test['route'] == route]
    
    # Global CI stats
    mean_ci_width_global = route_data['ridge_ci_width_global'].mean()
    coverage_global_route = ((route_data['delay_rate'] >= route_data['ridge_lower_global']) & 
                             (route_data['delay_rate'] <= route_data['ridge_upper_global'])).mean()
    
    # Route-specific CI stats
    ci_width_route = 2 * route_data['ci_halfwidth'].iloc[0]  # Full width
    coverage_route_route = ((route_data['delay_rate'] >= route_data['ridge_lower_route']) & 
                            (route_data['delay_rate'] <= route_data['ridge_upper_route'])).mean()
    
    # Model performance
    r2 = r2_score(route_data['delay_rate'], route_data['ridge_pred'])
    rmse = np.sqrt(mean_squared_error(route_data['delay_rate'], route_data['ridge_pred']))
    
    route_ci_stats.append({
        'route': route,
        'n': len(route_data),
        'r2': r2,
        'rmse': rmse,
        'ci_width_global': mean_ci_width_global,
        'coverage_global': coverage_global_route,
        'ci_width_route': ci_width_route,
        'coverage_route': coverage_route_route
    })

route_ci_df = pd.DataFrame(route_ci_stats)

# Define reliability thresholds
CI_RELIABLE_THRESHOLD = 0.10  # CI width < 10% = reliable
CI_CAUTION_THRESHOLD = 0.15   # CI width > 15% = needs caution

route_ci_df['reliability'] = route_ci_df['ci_width_route'].apply(
    lambda x: 'Reliable' if x < CI_RELIABLE_THRESHOLD else ('Caution' if x > CI_CAUTION_THRESHOLD else 'Moderate')
)

print("Route Prediction Reliability (sorted by CI width):")
print("=" * 110)
print(f"{'Route':<22} {'R²':>8} {'RMSE':>8} {'CI Width':>10} {'Coverage':>10} {'Reliability':>12}")
print("-" * 110)

for _, row in route_ci_df.sort_values('ci_width_route').iterrows():
    reliability_symbol = '✓' if row['reliability'] == 'Reliable' else ('⚠️' if row['reliability'] == 'Caution' else '~')
    print(f"{row['route']:<22} {row['r2']:>8.4f} {row['rmse']:>8.4f} {row['ci_width_route']:>10.4f} {row['coverage_route']*100:>9.1f}% {reliability_symbol} {row['reliability']:<10}")

In [ ]:
# Visualise route reliability
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Sort by CI width
route_ci_sorted = route_ci_df.sort_values('ci_width_route')
routes = route_ci_sorted['route'].values
x = np.arange(len(routes))

# Plot 1: CI width by route with reliability coloring
ax = axes[0]
colors = ['#27ae60' if r == 'Reliable' else ('#e74c3c' if r == 'Caution' else '#f39c12') 
          for r in route_ci_sorted['reliability']]
bars = ax.bar(x, route_ci_sorted['ci_width_route'], color=colors, alpha=0.8)

ax.axhline(CI_RELIABLE_THRESHOLD, color='green', linestyle='--', alpha=0.7, label=f'Reliable threshold ({CI_RELIABLE_THRESHOLD})')
ax.axhline(CI_CAUTION_THRESHOLD, color='red', linestyle='--', alpha=0.7, label=f'Caution threshold ({CI_CAUTION_THRESHOLD})')

ax.set_xticks(x)
ax.set_xticklabels([r.replace('_', '→') for r in routes], rotation=45, ha='right')
ax.set_ylabel('90% CI Width')
ax.set_title('Prediction Uncertainty by Route')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3, axis='y')

# Plot 2: R² vs CI width scatter
ax = axes[1]
colors = ['#27ae60' if r == 'Reliable' else ('#e74c3c' if r == 'Caution' else '#f39c12') 
          for r in route_ci_df['reliability']]
ax.scatter(route_ci_df['ci_width_route'], route_ci_df['r2'], c=colors, s=100, alpha=0.7, edgecolor='black')

# Label points
for _, row in route_ci_df.iterrows():
    ax.annotate(row['route'].replace('_', '→'), 
                (row['ci_width_route'], row['r2']),
                fontsize=7, ha='center', va='bottom', rotation=0)

ax.axvline(CI_RELIABLE_THRESHOLD, color='green', linestyle='--', alpha=0.5)
ax.axvline(CI_CAUTION_THRESHOLD, color='red', linestyle='--', alpha=0.5)
ax.axhline(0, color='black', linestyle='-', linewidth=0.5)

ax.set_xlabel('90% CI Width')
ax.set_ylabel('R²')
ax.set_title('Model Performance vs Uncertainty')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics by reliability category
print("Summary by Reliability Category:")
print("=" * 60)

for reliability in ['Reliable', 'Moderate', 'Caution']:
    subset = route_ci_df[route_ci_df['reliability'] == reliability]
    if len(subset) > 0:
        routes_list = ', '.join(subset['route'].str.replace('_', '→').tolist())
        print(f"\n{reliability.upper()} ({len(subset)} routes):")
        print(f"  Routes: {routes_list}")
        print(f"  Mean R²: {subset['r2'].mean():.4f}")
        print(f"  Mean CI width: {subset['ci_width_route'].mean():.4f}")
        print(f"  Mean coverage: {subset['coverage_route'].mean()*100:.1f}%")

## 13. Summary and Observations

In [ ]:
print("="*80)
print("FINAL MODEL SUMMARY WITH CONFIDENCE INTERVALS")
print("="*80)

# Count reliability categories
n_reliable = (route_ci_df['reliability'] == 'Reliable').sum()
n_moderate = (route_ci_df['reliability'] == 'Moderate').sum()
n_caution = (route_ci_df['reliability'] == 'Caution').sum()

# Get route lists
reliable_routes = route_ci_df[route_ci_df['reliability'] == 'Reliable']['route'].str.replace('_', '→').tolist()
caution_routes = route_ci_df[route_ci_df['reliability'] == 'Caution']['route'].str.replace('_', '→').tolist()

print(f"""
REGRESSION PERFORMANCE (Test):
  Ridge:         R² = {ridge_test_r2:.4f}, RMSE = {ridge_test_rmse:.4f}
  Random Forest: R² = {rf_test_r2:.4f}, RMSE = {rf_test_rmse:.4f}

CLASSIFICATION PERFORMANCE (Test):
  Logistic:      F1 = {logreg_test_f1:.4f}, AUC = {logreg_test_auc:.4f}
  Random Forest: F1 = {rf_clf_test_f1:.4f}, AUC = {rf_clf_test_auc:.4f}""")

if HAS_XGB:
    print(f"  XGBoost:       F1 = {xgb_test_f1:.4f}, AUC = {xgb_test_auc:.4f}")

print(f"""
CONFIDENCE INTERVALS (90% CI):
  Mean CI width (route-specific): {route_ci_df['ci_width_route'].mean():.4f}
  Overall coverage:               {coverage_route*100:.1f}%
  
  CI > {CI_CAUTION_THRESHOLD}:    {n_caution} routes
""")

### Observations:

- Overall prediction interval (90% CI) of 0.28, which corresponds to about +-14% uncertainty.
  - In layman's term, 90% of the prediction falls between +-14% delay rate interval. 
- Uncertainty is route-dependent, some routes had relatively narrow intervals (+-10%), while some have wider intervals (up to about +-20%). 
  - Routes with wider prediction intervals (higher uncertainty) should be treated with caution

## 13. Next Step

The next nextbook will test using dense neural network architecture to predict the delay rate (both regression and classification).